In [1]:
import sys
from pathlib import Path

# I had problems with relative imports in Jupyter notebooks, so I added the project root to sys.path
PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Added to sys.path:", PROJECT_ROOT)

Added to sys.path: C:\Users\gait2\Renewables in electricity markets\RES---Assignment-1


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from src.data_loader import load_rts24_from_csv
from src.models import Step1InputData, Step1MarketClearing, Step2InputData, Step2MarketClearing, Step3InputData, Step3MarketClearing, Step3ZonalInputData, Step3ZonalMarketClearing, Step5InputData, Step5BalancingMarket
from src.postprocess import step1_build_summary_tables, step2_build_summary_tables, step3_build_summary_tables, step3_zonal_build_summary_tables, step5_build_summary_tables

Define data directory

In [3]:
DATA_DIR = Path("..") / "data"

data = load_rts24_from_csv(DATA_DIR)

print(f"Slack bus: {data.slack_bus}")
print(f"Number of generators: {len(data.generators)}")
print(f"Number of lines: {len(data.lines)}")

Slack bus: 13
Number of generators: 12
Number of lines: 34


# STEP 1

Choose hour for copper-plate market clearing

In [4]:
H = 19  # peak hour example

Extract hour-specific data

In [5]:
# Generators
gen_df = data.generators.copy()

GENERATORS = gen_df["unit"].tolist()
generator_cost = dict(zip(gen_df["unit"], gen_df["Ci_USD_per_MWh"]))
generator_pmax = dict(zip(gen_df["unit"], gen_df["Pmax_MW"]))

# Wind availability
wind_farms_df = data.wind_farms.copy()
wind_av_h = data.wind_availability.query("hour == @H")

WINDS = wind_farms_df["wind_id"].tolist()
wind_avail = dict(zip(wind_av_h["wind_id"], wind_av_h["available_MW"]))


# Nodal demand
nodal_load_h = data.nodal_load.query("hour == @H")

LOAD_BUSES = nodal_load_h.query("demand_MW > 0")["bus"].tolist()
demand_pmax = dict(zip(nodal_load_h["bus"], nodal_load_h["demand_MW"]))


# Demand bids
bids_h = data.demand_bids.query("hour == @H")
demand_bid = dict(zip(bids_h["bus"], bids_h["bid_price_USD_per_MWh"]))

Build input object

In [6]:
inp = Step1InputData(
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    wind_avail=wind_avail,
    demand_pmax=demand_pmax,
    demand_bid=demand_bid,
)

Solve Step 1 (Copper Plate Market Clearing)

In [7]:
model = Step1MarketClearing(inp, output_flag=0)
model.run()

Restricted license - for non-production use only - expires 2026-11-23


**Results**

In [8]:
df_gen, df_wind, df_dem, totals = step1_build_summary_tables(model.results, inp)

print("Market price:", round(totals["market_price"], 2), "USD/MWh")
print("Objective (Social Welfare):", round(totals["objective_welfare"], 2))
print("Total operating cost:", round(totals["total_operating_cost"], 2))

print("\nTotal generation:", round(totals["total_gen_MW"], 2), "MW")
print("Total wind:", round(totals["total_wind_MW"], 2), "MW")
print("Total demand served:", round(totals["total_demand_served_MW"], 2), "MW")

Market price: 13.32 USD/MWh
Objective (Social Welfare): 619868.95
Total operating cost: 16251.05

Total generation: 2169.19 MW
Total wind: 481.31 MW
Total demand served: 2650.5 MW


Generator results

In [9]:
df_gen

,generator,p_MW,marginal_cost,profit
0,1,99.185514,13.32,0.0
1,2,0.000000,13.32,0.0
2,3,0.000000,20.70,-0.0
3,4,0.000000,20.93,-0.0
4,5,0.000000,26.11,-0.0
5,6,155.000000,10.52,434.0
6,7,155.000000,10.52,434.0
7,8,400.000000,6.02,2920.0
8,9,400.000000,5.47,3140.0
9,10,300.000000,0.00,3996.0


Wind results

In [10]:
df_wind

,wind,p_MW,profit
0,1,73.622733,980.654803
1,2,83.964245,1118.403746
2,3,80.763464,1075.769341
3,4,82.379405,1097.293675
4,5,73.687627,981.519194
5,6,86.897011,1157.468190


Demand results

In [11]:
df_dem

,bus,d_MW,bid_price,utility
0,1,100.7190,240.0,22830.98292
1,2,90.1170,240.0,20427.72156
2,3,166.9815,240.0,37851.36642
3,4,68.9130,240.0,15621.19884
4,5,66.2625,240.0,15020.38350
5,6,127.2240,240.0,28839.13632
6,7,116.6220,240.0,26435.87496
7,8,159.0300,240.0,36048.92040
8,9,161.6805,240.0,36649.73574
9,10,180.2340,240.0,40855.44312


Welfare breakdown check

In [12]:
print("Total generator profit:", round(totals["total_gen_profit"], 2))
print("Total wind profit:", round(totals["total_wind_profit"], 2))
print("Total demand utility:", round(totals["total_demand_utility"], 2))

print("\nCheck: Utility - Cost = Welfare")
print(round(
    totals["total_demand_utility"]
    + totals["total_gen_profit"]
    + totals["total_wind_profit"],
    6
))

Total generator profit: 12642.5
Total wind profit: 6411.11
Total demand utility: 600815.34

Check: Utility - Cost = Welfare
619868.948948


# STEP 2

In [13]:
# Generators
gen_df = data.generators.copy()

GENERATORS = gen_df["unit"].tolist()
generator_cost = dict(zip(gen_df["unit"], gen_df["Ci_USD_per_MWh"]))
generator_pmax = dict(zip(gen_df["unit"], gen_df["Pmax_MW"]))

# Hours (all 24)
hours = [int(h) for h in sorted(data.wind_availability["hour"].unique())]

# Wind availability
wind_farms_df = data.wind_farms.copy()
WINDS = wind_farms_df["wind_id"].tolist()

wind_avail_t = {}

for h in hours:
    wind_av_h = data.wind_availability.query("hour == @h")
    wind_avail_t[h] = dict(zip(wind_av_h["wind_id"], wind_av_h["available_MW"]))


# Nodal demand
LOAD_BUSES = data.nodal_load.query("demand_MW > 0")["bus"].unique().tolist()

demand_pmax_t = {}
for h in hours:
    nodal_load_h = data.nodal_load.query("hour == @h")
    demand_pmax_t[h] = dict(zip(nodal_load_h["bus"], nodal_load_h["demand_MW"]))


# Demand bids
demand_bid_t = {}
for h in hours:
    bids_h = data.demand_bids.query("hour == @h")
    demand_bid_t[h] = dict(zip(bids_h["bus"], bids_h["bid_price_USD_per_MWh"]))

In [14]:
inpu = Step2InputData(
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    wind_avail_t=wind_avail_t,
    demand_pmax_t=demand_pmax_t,
    demand_bid_t=demand_bid_t,
    hours=hours                   
)
print("hours:", hours, type(hours))


hours: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24] <class 'list'>


In [15]:
model2 = Step2MarketClearing(inpu, output_flag=0)
model2.run()

In [16]:
df_gen, df_wind, df_dem, df_storage, df_prices, totals = step2_build_summary_tables(model2.results, inpu)


# Results

In [17]:
print("\n1. HOURLY PRICES")
print(df_prices.round(2))


1. HOURLY PRICES
    hour  price
0      1  10.52
1      2  10.52
2      3  10.52
3      4  10.52
4      5  10.52
5      6  10.52
6      7  10.52
7      8  10.89
8      9  10.89
9     10  10.89
10    11  10.89
11    12  10.89
12    13  10.89
13    14  10.89
14    15  10.89
15    16  10.89
16    17  12.57
17    18  12.57
18    19  12.57
19    20  10.89
20    21  10.89
21    22  10.52
22    23  10.52
23    24  10.52


In [18]:
print("\n2. TOTALS 24 HOURS")
print(pd.Series(totals).round(2))


2. TOTALS 24 HOURS
objective_welfare_24h       11211650.47
total_operating_cost_24h      257593.13
total_gen_MWh                  40311.46
total_wind_MWh                 12506.05
total_demand_served_MWh        52771.46
total_gen_profit_24h          186758.31
total_wind_profit_24h         137151.04
total_demand_utility_24h    10887741.11
total_storage_profit_24h           0.00
avg_price                         10.95
max_price                         12.57
min_price                         10.52
dtype: float64


In [19]:
print("\n3. GENERATOR TOTALS & PROFITS")
print(df_gen.round(2))


3. GENERATOR TOTALS & PROFITS
    generator  total_p_MWh  marginal_cost  total_profit_24h
0           1         0.00          13.32              0.00
1           2         0.00          13.32              0.00
2           3         0.00          20.70              0.00
3           4         0.00          20.93              0.00
4           5         0.00          26.11              0.00
5           6      2758.52          10.52           1583.49
6           7      2536.38          10.52           1583.49
7           8      9600.00           6.02          47286.44
8           9      9600.00           5.47          52566.44
9          10      7200.00           0.00          78808.83
10         11      6074.41          10.52           3166.99
11         12      2542.14          10.89           1762.63


In [20]:
print("\n4. WIND TOTALS & PROFITS") 
print(df_wind.round(2))


4. WIND TOTALS & PROFITS
   wind  total_p_MWh  total_profit_24h
0     1      2090.17          22901.39
1     2      2062.08          22615.25
2     3      2045.86          22434.50
3     4      2130.81          23381.73
4     5      2072.97          22725.50
5     6      2104.17          23092.67


In [21]:
print("\n5. STORAGE OPERATION (Hourly)")
print(df_storage.round(2))


5. STORAGE OPERATION (Hourly)
    hour  pch_MW  pdis_MW   e_MWh  price  profit_t
0      1    0.00     0.00    0.00  10.52      0.00
1      2    0.00     0.00    0.00  10.52      0.00
2      3    0.00     0.00    0.00  10.52      0.00
3      4    0.00     0.00    0.00  10.52      0.00
4      5  282.54     0.00  254.29  10.52  -2972.31
5      6    0.00     0.00  254.29  10.52      0.00
6      7    0.00     0.00  254.29  10.52      0.00
7      8    0.00     0.00  254.29  10.89      0.00
8      9    0.00     0.00  254.29  10.89      0.00
9     10    0.00     0.00  254.29  10.89      0.00
10    11    0.00     0.00  254.29  10.89      0.00
11    12    0.00     0.00  254.29  10.89      0.00
12    13    0.00     0.00  254.29  10.89      0.00
13    14    0.00     0.00  254.29  10.89      0.00
14    15    0.00     0.00  254.29  10.89      0.00
15    16    0.00     0.00  254.29  10.89      0.00
16    17    0.00    68.26  180.89  12.57    857.96
17    18    0.00    69.04  106.65  12.57    867.72


In [22]:
print("\n6. DEMAND TOTALS & UTILITY")
print(df_dem.round(2))


6. DEMAND TOTALS & UTILITY
    bus  total_d_MWh  total_utility_24h
0     1      2005.32          413734.16
1     2      1794.23          370183.20
2     3      3324.60          685927.69
3     4      1372.06          283081.27
4     5      1319.29          272193.53
5     6      2533.03          522611.57
6     7      2321.94          479060.61
7     8      3166.29          653264.47
8     9      3219.06          664152.21
9    10      3588.46          740366.40
10   13      4907.75         1012559.92
11   14      3588.46          740366.40
12   15      5857.63         1208539.26
13   16      1847.00          381070.94
14   18      6174.26         1273865.71
15   19      3377.37          696815.43
16   20      2374.72          489948.35


# STEP 3

In [23]:
# STEP 3A - Build nodal input data


H = 19   # same representative hour as Step 1

# Generators
gen_df = data.generators.copy()
GENERATORS = gen_df["unit"].tolist()
generator_cost = dict(zip(gen_df["unit"], gen_df["Ci_USD_per_MWh"]))
generator_pmax = dict(zip(gen_df["unit"], gen_df["Pmax_MW"]))
generator_bus = dict(zip(gen_df["unit"], gen_df["bus"]))

# Wind
wind_farms_df = data.wind_farms.copy()
wind_av_h = data.wind_availability.query("hour == @H")

WINDS = wind_farms_df["wind_id"].tolist()
wind_avail = dict(zip(wind_av_h["wind_id"], wind_av_h["available_MW"]))
wind_bus = dict(zip(wind_farms_df["wind_id"], wind_farms_df["bus"]))

# Demand
nodal_load_h = data.nodal_load.query("hour == @H")
LOAD_BUSES = nodal_load_h.query("demand_MW > 0")["bus"].tolist()
demand_pmax = dict(zip(nodal_load_h["bus"], nodal_load_h["demand_MW"]))

bids_h = data.demand_bids.query("hour == @H")
demand_bid = dict(zip(bids_h["bus"], bids_h["bid_price_USD_per_MWh"]))

# Buses
BUSES = data.buses

# Lines
lines_df = data.lines.copy().reset_index(drop=True)
LINES = lines_df.index.tolist()

line_from = dict(zip(LINES, lines_df["from_bus"]))
line_to = dict(zip(LINES, lines_df["to_bus"]))
line_B = {l: 1 / lines_df.loc[l, "reactance_pu"] for l in LINES}
line_fmax = dict(zip(LINES, lines_df["capacity_modified_MW"]))

inp3 = Step3InputData(
    BUSES=BUSES,
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    LINES=LINES,
    slack_bus=data.slack_bus,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    generator_bus=generator_bus,
    wind_avail=wind_avail,
    wind_bus=wind_bus,
    demand_pmax=demand_pmax,
    demand_bid=demand_bid,
    line_from=line_from,
    line_to=line_to,
    line_B=line_B,
    line_fmax=line_fmax,
)

print("Buses:", len(BUSES))
print("Lines:", len(LINES))
print("Slack bus:", data.slack_bus)
print("Example susceptance:", round(line_B[0], 3))
print("Example capacity:", line_fmax[0])

Buses: 24
Lines: 34
Slack bus: 13
Example susceptance: 68.493
Example capacity: 175.0


In [24]:
# STEP 3A - Solve nodal model


model3 = Step3MarketClearing(inp3, output_flag=0)
model3.run()

df_gen3, df_wind3, df_dem3, df_bus3, df_line3, totals3 = step3_build_summary_tables(model3.results, inp3)

print("STEP 3A - NODAL MARKET CLEARING")
print("Objective (Social Welfare):", round(totals3["objective_welfare"], 2))
print("Total operating cost:", round(totals3["total_operating_cost"], 2))
print("Total generation:", round(totals3["total_gen_MW"], 2), "MW")
print("Total wind:", round(totals3["total_wind_MW"], 2), "MW")
print("Total demand served:", round(totals3["total_demand_served_MW"], 2), "MW")
print("Min nodal price:", round(totals3["min_nodal_price"], 2), "USD/MWh")
print("Max nodal price:", round(totals3["max_nodal_price"], 2), "USD/MWh")
print("Price spread:", round(totals3["price_spread"], 2), "USD/MWh")
print("Congested lines:", totals3["n_congested_lines"])

STEP 3A - NODAL MARKET CLEARING
Objective (Social Welfare): 615455.7
Total operating cost: 20664.3
Total generation: 2169.19 MW
Total wind: 481.31 MW
Total demand served: 2650.5 MW
Min nodal price: 5.47 USD/MWh
Max nodal price: 29.45 USD/MWh
Price spread: 23.98 USD/MWh
Congested lines: 3


In [25]:
# display nodal results
df_bus3.round(3)

,bus,theta_rad,nodal_price,gen_MW,wind_MW,demand_MW,net_injection_MW
0,1,-2.254,19.709,152.000,0.000,100.719,51.281
1,2,-2.713,19.809,152.000,0.000,90.117,61.883
2,3,-1.803,16.545,0.000,73.623,166.982,-93.359
3,4,-8.625,20.109,0.000,0.000,68.913,-68.913
4,5,-4.231,20.365,0.000,83.964,66.262,17.702
5,6,-12.916,20.750,0.000,0.000,127.224,-127.224
6,7,14.233,20.700,264.972,80.763,116.622,229.113
7,8,-0.705,20.700,0.000,0.000,159.030,-159.030
8,9,-5.815,20.355,0.000,0.000,161.680,-161.680
9,10,-7.944,21.045,0.000,0.000,180.234,-180.234


In [26]:
df_line3.round(3)

,line,from_bus,to_bus,B,flow_MW,Fmax_MW,loading_pct,congested
0,0,1,2,68.493,31.484,175.0,17.991,False
1,1,11,13,20.492,-43.492,500.0,8.698,False
2,2,1,3,4.439,-2.000,175.0,1.143,False
3,3,11,14,23.474,-69.766,500.0,13.953,False
4,4,1,5,11.025,21.797,350.0,6.228,False
5,5,12,13,20.492,22.795,500.0,4.559,False
6,6,2,4,7.375,43.597,175.0,24.913,False
7,7,12,23,10.152,-213.072,500.0,42.614,False
8,8,2,6,4.878,49.770,175.0,28.440,False
9,9,13,23,11.312,-250.000,250.0,100.000,True


In [27]:
df_gen3.round(3)

,generator,bus,p_MW,marginal_cost,nodal_price,profit
0,1,1,152.000,13.32,19.709,971.155
1,2,2,152.000,13.32,19.809,986.265
2,3,7,264.972,20.70,20.700,0.000
3,4,13,17.194,20.93,20.930,0.000
4,5,15,0.000,26.11,10.520,-0.000
5,6,15,13.786,10.52,10.520,0.000
6,7,16,0.000,10.52,8.666,-0.000
7,8,18,400.000,6.02,6.198,71.374
8,9,21,209.234,5.47,5.470,0.000
9,10,22,300.000,0.00,6.066,1819.903


In [28]:
df_wind3.round(3)

,wind,bus,p_MW,nodal_price,profit
0,1,3,73.623,16.545,1218.120
1,2,5,83.964,20.365,1709.955
2,3,7,80.763,20.700,1671.804
3,4,16,82.379,8.666,713.865
4,5,21,73.688,5.470,403.071
5,6,23,86.897,13.528,1175.547


In [29]:
df_dem3.round(3)

,bus,d_MW,bid_price,nodal_price,utility
0,1,100.719,240.0,19.709,22187.472
1,2,90.117,240.0,19.809,19842.990
2,3,166.982,240.0,16.545,37312.778
3,4,68.913,240.0,20.109,15153.354
4,5,66.262,240.0,20.365,14553.546
5,6,127.224,240.0,20.750,27893.822
6,7,116.622,240.0,20.700,25575.205
7,8,159.030,240.0,20.700,34875.279
8,9,161.680,240.0,20.355,35512.351
9,10,180.234,240.0,21.045,39463.093


In [30]:
# nodal sensitivity analysis

from copy import deepcopy

def run_step3_line_sensitivity(inp_base, tested_line, tested_capacities):
    rows = []

    for cap in tested_capacities:
        inp_tmp = deepcopy(inp_base)
        inp_tmp.line_fmax[tested_line] = cap

        model_tmp = Step3MarketClearing(inp_tmp, output_flag=0)
        model_tmp.run()

        _, _, _, _, _, totals_tmp = step3_build_summary_tables(model_tmp.results, inp_tmp)

        rows.append({
            "line": tested_line,
            "tested_capacity_MW": cap,
            "min_nodal_price": totals_tmp["min_nodal_price"],
            "max_nodal_price": totals_tmp["max_nodal_price"],
            "price_spread": totals_tmp["price_spread"],
            "social_welfare": totals_tmp["objective_welfare"],
            "n_congested_lines": totals_tmp["n_congested_lines"],
        })

    return pd.DataFrame(rows)

#To identify the line index for one of your tightened lines
lines_df[["from_bus", "to_bus", "capacity_modified_MW"]]

# Then run, for example, on one of the modified lines 

sens_df = run_step3_line_sensitivity(
    inp_base=inp3,
    tested_line=11,  #line 14-16
    tested_capacities=[150, 200, 250, 300, 400]
)

sens_df.round(3)

,line,tested_capacity_MW,min_nodal_price,max_nodal_price,price_spread,social_welfare,n_congested_lines
0,11,150,4.831,35.751,30.920,611467.555,1
1,11,200,5.548,35.090,29.541,613611.796,1
2,11,250,5.470,29.447,23.977,615455.699,3
3,11,300,5.470,25.032,19.562,616882.518,3
4,11,400,5.470,20.930,15.460,617547.900,2


In [31]:
# STEP 3B - Build zonal input data
ZONES = ["Z1", "Z2"]
BUS_ZONE = {b: ("Z1" if b <= 15 else "Z2") for b in BUSES}

# One interface between the two zones
INTERFACES = [("Z1", "Z2")]

# ATC = total capacity of all crossing lines (allowed by assignment)
crossing_lines = [
    l for l in LINES
    if BUS_ZONE[line_from[l]] != BUS_ZONE[line_to[l]]
]

atc = {
    ("Z1", "Z2"): sum(line_fmax[l] for l in crossing_lines)
}

print("Crossing lines:", crossing_lines)
print("ATC Z1-Z2:", atc[("Z1", "Z2")], "MW")

inp3z = Step3ZonalInputData(
    ZONES=ZONES,
    BUS_ZONE=BUS_ZONE,
    INTERFACES=INTERFACES,
    atc=atc,
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    generator_bus=generator_bus,
    wind_avail=wind_avail,
    wind_bus=wind_bus,
    demand_pmax=demand_pmax,
    demand_bid=demand_bid,
)

Crossing lines: [7, 9, 11, 12, 13, 15, 17]
ATC Z1-Z2: 2800.0 MW


In [32]:
# STEP 3B - Solve zonal model


model3z = Step3ZonalMarketClearing(inp3z, output_flag=0)
model3z.run()

df_gen3z, df_wind3z, df_dem3z, df_zone3z, df_transfer3z, totals3z = step3_zonal_build_summary_tables(model3z.results, inp3z)

print("STEP 3B - ZONAL MARKET CLEARING")
print("Objective (Social Welfare):", round(totals3z["objective_welfare"], 2))
print("Total operating cost:", round(totals3z["total_operating_cost"], 2))
print("Total generation:", round(totals3z["total_gen_MW"], 2), "MW")
print("Total wind:", round(totals3z["total_wind_MW"], 2), "MW")
print("Total demand served:", round(totals3z["total_demand_served_MW"], 2), "MW")
print("Zonal prices:", {z: round(model3z.results.zonal_prices[z], 2) for z in ZONES})

df_zone3z.round(3)
df_transfer3z.round(3)
df_gen3z.round(3)
df_wind3z.round(3)
df_dem3z.round(3)

STEP 3B - ZONAL MARKET CLEARING
Objective (Social Welfare): 619868.95
Total operating cost: 16251.05
Total generation: 2169.19 MW
Total wind: 481.31 MW
Total demand served: 2650.5 MW
Zonal prices: {'Z1': 13.32, 'Z2': 13.32}


,bus,zone,d_MW,bid_price,zonal_price,utility
0,1,Z1,100.719,240.0,13.32,22830.983
1,2,Z1,90.117,240.0,13.32,20427.722
2,3,Z1,166.982,240.0,13.32,37851.366
3,4,Z1,68.913,240.0,13.32,15621.199
4,5,Z1,66.262,240.0,13.32,15020.384
5,6,Z1,127.224,240.0,13.32,28839.136
6,7,Z1,116.622,240.0,13.32,26435.875
7,8,Z1,159.030,240.0,13.32,36048.920
8,9,Z1,161.680,240.0,13.32,36649.736
9,10,Z1,180.234,240.0,13.32,40855.443


In [33]:
compare_df = pd.DataFrame([
    {
        "framework": "Nodal",
        "social_welfare": totals3["objective_welfare"],
        "total_gen_profit": totals3["total_gen_profit"],
        "total_wind_profit": totals3["total_wind_profit"],
        "total_demand_utility": totals3["total_demand_utility"],
        "price_info": f'{round(totals3["min_nodal_price"],2)} to {round(totals3["max_nodal_price"],2)}'
    },
    {
        "framework": "Zonal",
        "social_welfare": totals3z["objective_welfare"],
        "total_gen_profit": totals3z["total_gen_profit"],
        "total_wind_profit": totals3z["total_wind_profit"],
        "total_demand_utility": totals3z["total_demand_utility"],
        "price_info": str({z: round(model3z.results.zonal_prices[z], 2) for z in ZONES})
    }
])

compare_df

,framework,social_welfare,total_gen_profit,total_wind_profit,total_demand_utility,price_info
0,Nodal,615455.699274,5704.506554,6892.361286,591817.448586,5.47 to 29.45
1,Zonal,619868.948948,12642.500000,6411.108948,600815.340000,"{'Z1': 13.32, 'Z2': 13.32}"


In [34]:
def dc_flow_from_injections(inp_nodal, injections):
    """
    Solve DC load flow from fixed net injections using the original nodal network.
    """
    buses = inp_nodal.BUSES
    idx = {b: i for i, b in enumerate(buses)}
    n = len(buses)

    Bbus = np.zeros((n, n))

    for l in inp_nodal.LINES:
        i = idx[inp_nodal.line_from[l]]
        j = idx[inp_nodal.line_to[l]]
        b = inp_nodal.line_B[l]

        Bbus[i, i] += b
        Bbus[j, j] += b
        Bbus[i, j] -= b
        Bbus[j, i] -= b

    slack_i = idx[inp_nodal.slack_bus]
    keep = [i for i in range(n) if i != slack_i]

    P = np.array([injections.get(b, 0.0) for b in buses], dtype=float)

    theta = np.zeros(n)
    theta_red = np.linalg.solve(Bbus[np.ix_(keep, keep)], P[keep])
    theta[keep] = theta_red

    flows = {}
    for l in inp_nodal.LINES:
        i = idx[inp_nodal.line_from[l]]
        j = idx[inp_nodal.line_to[l]]
        flows[l] = inp_nodal.line_B[l] * (theta[i] - theta[j])

    theta_dict = {b: theta[idx[b]] for b in buses}
    return theta_dict, flows

zonal_injections = {}
for b in BUSES:
    gen_b = sum(model3z.results.pG[g] for g in GENERATORS if generator_bus[g] == b)
    wind_b = sum(model3z.results.pW[w] for w in WINDS if wind_bus[w] == b)
    dem_b = model3z.results.d[b] if b in LOAD_BUSES else 0.0
    zonal_injections[b] = gen_b + wind_b - dem_b

theta_zonal_dc, flows_zonal_dc = dc_flow_from_injections(inp3, zonal_injections)

feas_rows = []
for l in LINES:
    overload = max(0.0, abs(flows_zonal_dc[l]) - line_fmax[l])
    same_zone = BUS_ZONE[line_from[l]] == BUS_ZONE[line_to[l]]

    feas_rows.append({
        "line": l,
        "from_bus": line_from[l],
        "to_bus": line_to[l],
        "same_zone": same_zone,
        "flow_from_zonal_dispatch_MW": flows_zonal_dc[l],
        "Fmax_MW": line_fmax[l],
        "overload_MW": overload
    })

df_feas = pd.DataFrame(feas_rows)
df_internal_overload = df_feas[(df_feas["same_zone"]) & (df_feas["overload_MW"] > 1e-5)].copy()

print("Total overloaded internal lines:", len(df_internal_overload))
print("Max internal overload (MW):", round(df_internal_overload["overload_MW"].max(), 4) if len(df_internal_overload) > 0 else 0.0)
print("Sum of internal overloads (MW):", round(df_internal_overload["overload_MW"].sum(), 4))

df_internal_overload.round(3)

Total overloaded internal lines: 0
Max internal overload (MW): 0.0
Sum of internal overloads (MW): 0.0


,line,from_bus,to_bus,same_zone,flow_from_zonal_dispatch_MW,Fmax_MW,overload_MW


In [35]:
df_all_overload = df_feas[df_feas["overload_MW"] > 1e-5].copy()

print("Total overloaded lines:", len(df_all_overload))
print("Max overload (MW):", round(df_all_overload["overload_MW"].max(), 4) if len(df_all_overload) > 0 else 0.0)
print("Sum of overloads (MW):", round(df_all_overload["overload_MW"].sum(), 4))

df_all_overload.round(3)

Total overloaded lines: 3
Max overload (MW): 172.6309
Sum of overloads (MW): 356.2427


,line,from_bus,to_bus,same_zone,flow_from_zonal_dispatch_MW,Fmax_MW,overload_MW
9,9,13,23,False,-330.841,250.0,80.841
11,11,14,16,False,-422.631,250.0,172.631
15,15,15,21,False,-502.770,400.0,102.770


In [36]:
# STEP 3B — ATC Sensitivity Analysis

atc_values = [400, 800, 1200, 2800]

zonal_sensitivity_rows = []

for atc_val in atc_values:
    inp3z_tmp = Step3ZonalInputData(
        ZONES=ZONES,
        BUS_ZONE=BUS_ZONE,
        INTERFACES=INTERFACES,
        atc={("Z1", "Z2"): atc_val},
        GENERATORS=GENERATORS,
        WINDS=WINDS,
        LOAD_BUSES=LOAD_BUSES,
        generator_cost=generator_cost,
        generator_pmax=generator_pmax,
        generator_bus=generator_bus,
        wind_avail=wind_avail,
        wind_bus=wind_bus,
        demand_pmax=demand_pmax,
        demand_bid=demand_bid,
    )

    model3z_tmp = Step3ZonalMarketClearing(inp3z_tmp, output_flag=0)
    model3z_tmp.run()

    _, _, _, _, _, totals3z_tmp = step3_zonal_build_summary_tables(model3z_tmp.results, inp3z_tmp)

    # Ex-post feasibility check
    zonal_inj_tmp = {}
    for b in BUSES:
        gen_b = sum(model3z_tmp.results.pG[g] for g in GENERATORS if generator_bus[g] == b)
        wind_b = sum(model3z_tmp.results.pW[w] for w in WINDS if wind_bus[w] == b)
        dem_b = model3z_tmp.results.d[b] if b in LOAD_BUSES else 0.0
        zonal_inj_tmp[b] = gen_b + wind_b - dem_b

    _, flows_tmp = dc_flow_from_injections(inp3, zonal_inj_tmp)

    total_overload = sum(
        max(0.0, abs(flows_tmp[l]) - line_fmax[l]) for l in LINES
    )
    n_overloaded = sum(
        1 for l in LINES if abs(flows_tmp[l]) - line_fmax[l] > 1e-5
    )

    zonal_sensitivity_rows.append({
        "ATC_MW": atc_val,
        "price_Z1": model3z_tmp.results.zonal_prices["Z1"],
        "price_Z2": model3z_tmp.results.zonal_prices["Z2"],
        "social_welfare": totals3z_tmp["objective_welfare"],
        "total_gen_profit": totals3z_tmp["total_gen_profit"],
        "total_wind_profit": totals3z_tmp["total_wind_profit"],
        "total_demand_utility": totals3z_tmp["total_demand_utility"],
        "n_overloaded_lines": n_overloaded,
        "total_overload_MW": total_overload,
    })

df_atc_sens = pd.DataFrame(zonal_sensitivity_rows)
print("ZONAL ATC SENSITIVITY ANALYSIS")
print(df_atc_sens.round(2))

ZONAL ATC SENSITIVITY ANALYSIS
   ATC_MW  price_Z1  price_Z2  social_welfare  total_gen_profit  \
0     400     20.93      6.02       609408.29           6033.49   
1     800     20.93     10.52       614702.62          10983.49   
2    1200     20.70     10.89       618769.22          11376.47   
3    2800     13.32     13.32       619868.95          12642.50   

   total_wind_profit  total_demand_utility  n_overloaded_lines  \
0            6451.32             590959.48                   0   
1            7544.66             587846.47                   2   
2            7579.73             588041.02                   2   
3            6411.11             600815.34                   3   

   total_overload_MW  
0               0.00  
1             106.51  
2             221.85  
3             356.24  


# STEP 5

In [37]:
from src.models import Step5InputData, Step5BalancingMarket
from src.postprocess import step5_build_summary_tables

Scenario Definition:
Failed generator: G8 (largest DA dispatch at 400 MW → biggest shortage)
Wind deviations:  W1, W2, W3 → -15% of DA production
                   W4, W5, W6 → +10% of DA production
Load curtailment cost: 500 $/MWh

In [38]:
FAILED_GEN       = 8
WIND_DEV_PCT     = {1: -0.15, 2: -0.15, 3: -0.15,
                    4: +0.10, 5: +0.10, 6: +0.10}
LC_COST          = 500.0

λ_DA = model.results.market_price

Wind production

In [39]:
wind_da   = {w: model.results.pW[w] for w in inp.WINDS}
wind_real = {w: wind_da[w] * (1.0 + WIND_DEV_PCT[w]) for w in inp.WINDS}

print("Wind DA vs real-time production:")
for w in inp.WINDS:
    delta = wind_real[w] - wind_da[w]
    print(f"  W{w}: DA={wind_da[w]:.2f} MW  →  Real={wind_real[w]:.2f} MW  (Δ={delta:+.2f} MW)")

Wind DA vs real-time production:
  W1: DA=73.62 MW  →  Real=62.58 MW  (Δ=-11.04 MW)
  W2: DA=83.96 MW  →  Real=71.37 MW  (Δ=-12.59 MW)
  W3: DA=80.76 MW  →  Real=68.65 MW  (Δ=-12.11 MW)
  W4: DA=82.38 MW  →  Real=90.62 MW  (Δ=+8.24 MW)
  W5: DA=73.69 MW  →  Real=81.06 MW  (Δ=+7.37 MW)
  W6: DA=86.90 MW  →  Real=95.59 MW  (Δ=+8.69 MW)


System imbalance    

In [40]:
pG_failed      = model.results.pG[FAILED_GEN]
wind_delta     = sum(wind_real[w] - wind_da[w] for w in inp.WINDS)
# Positive = shortage (real supply < DA schedule → need upward regulation)
system_imbalance = pG_failed - wind_delta

print(f"\nG{FAILED_GEN} outage:       -{pG_failed:.2f} MW")
print(f"Wind net deviation:  {wind_delta:+.2f} MW")
print(f"System imbalance:    {system_imbalance:+.2f} MW  "
      f"({'shortage → upward regulation needed' if system_imbalance > 0 else 'surplus → downward regulation needed'})")


G8 outage:       -400.00 MW
Wind net deviation:  -11.46 MW
System imbalance:    +411.46 MW  (shortage → upward regulation needed)


Balancing offers

In [41]:
GENERATORS_UP, GENERATORS_DN = [], []
up_price, dn_price = {}, {}
up_cap,   dn_cap   = {}, {}

for g in inp.GENERATORS:
    if g == FAILED_GEN:
        continue                          # offline, cannot offer

    pG_DA = model.results.pG[g]
    Pmax  = inp.generator_pmax[g]
    c_g   = inp.generator_cost[g]

    headroom = Pmax  - pG_DA             # can ramp up to Pmax
    footroom = pG_DA                     # can reduce down to 0

    if headroom > 1e-6:
        GENERATORS_UP.append(g)
        up_price[g] = λ_DA + 0.10 * c_g
        up_cap[g]   = headroom

    if footroom > 1e-6:
        GENERATORS_DN.append(g)
        dn_price[g] = λ_DA - 0.15 * c_g
        dn_cap[g]   = footroom

print("\nUpward offer stack (sorted cheapest first):")
for g in sorted(GENERATORS_UP, key=lambda g: up_price[g]):
    print(f"  G{g}: {up_price[g]:.4f} $/MWh,  cap={up_cap[g]:.2f} MW")

print("\nDownward offer stack (sorted most expensive first):")
for g in sorted(GENERATORS_DN, key=lambda g: dn_price[g], reverse=True):
    print(f"  G{g}: {dn_price[g]:.4f} $/MWh,  cap={dn_cap[g]:.2f} MW")


Upward offer stack (sorted cheapest first):
  G1: 14.6520 $/MWh,  cap=52.81 MW
  G2: 14.6520 $/MWh,  cap=152.00 MW
  G3: 15.3900 $/MWh,  cap=350.00 MW
  G4: 15.4130 $/MWh,  cap=591.00 MW
  G5: 15.9310 $/MWh,  cap=60.00 MW

Downward offer stack (sorted most expensive first):
  G10: 13.3200 $/MWh,  cap=300.00 MW
  G9: 12.4995 $/MWh,  cap=400.00 MW
  G6: 11.7420 $/MWh,  cap=155.00 MW
  G7: 11.7420 $/MWh,  cap=155.00 MW
  G11: 11.7420 $/MWh,  cap=310.00 MW
  G12: 11.6865 $/MWh,  cap=350.00 MW
  G1: 11.3220 $/MWh,  cap=99.19 MW


Running the model and table building

In [42]:
inp5 = Step5InputData(
    GENERATORS_UP=GENERATORS_UP,
    GENERATORS_DN=GENERATORS_DN,
    up_price=up_price,
    dn_price=dn_price,
    up_cap=up_cap,
    dn_cap=dn_cap,
    system_imbalance=system_imbalance,
    load_curtailment_cost=LC_COST,
)

model5 = Step5BalancingMarket(inp5, output_flag=0)
model5.run()

print(f"\nBalancing price (λ_B):     {model5.results.balancing_price:.4f} $/MWh")
print(f"DA price        (λ_DA):    {λ_DA:.4f} $/MWh")
print(f"Load curtailment:          {model5.results.LC:.2f} MW")

df_offers5, df_gen5, df_wind5, totals5 = step5_build_summary_tables(
    step5_results   = model5.results,
    step5_input     = inp5,
    da_results      = model.results,
    da_input        = inp,
    failed_generator= FAILED_GEN,
    wind_da_dispatch= wind_da,
    wind_real       = wind_real,
    system_imbalance= system_imbalance,
)



Balancing price (λ_B):     15.3900 $/MWh
DA price        (λ_DA):    13.3200 $/MWh
Load curtailment:          0.00 MW


Results display and one-two price scheme interpretation

In [43]:
# DISPLAY RESULTS

print("\n=== BALANCING OFFER STACK & ACTIVATIONS ===")
print(df_offers5.round(3).to_string(index=False))
 
print("\n=== GENERATOR PROFITS: DA + BALANCING (1-price vs 2-price) ===")
print(df_gen5[["generator","role","pG_DA_MW","deviation_MW",
               "da_profit","balancing_profit_1p","total_profit_1p",
               "balancing_profit_2p","total_profit_2p"]].round(2).to_string(index=False))
 
print("\n=== WIND FARM PROFITS: 1-PRICE vs 2-PRICE ===")
print(df_wind5[["wind","pW_DA_MW","pW_real_MW","deviation_MW",
                "da_profit","total_profit_1price","total_profit_2price"]].round(2).to_string(index=False))
 
print("\n=== TOTALS ===")
print(pd.Series(totals5).round(4).to_string())
 

# ONE-PRICE vs TWO-PRICE FULL COMPARISON (BRPs only)

print("\n=== ONE-PRICE vs TWO-PRICE: FULL BRP ANALYSIS ===")
λ_B = model5.results.balancing_price
sys_dir = "SHORT" if system_imbalance > 0 else "LONG"
print(f"System is {sys_dir}.  λ_DA={λ_DA:.2f} $/MWh,  λ_B={λ_B:.2f} $/MWh\n")
 
print("--- Generators (BRPs and BSPs) ---")
for _, row in df_gen5.iterrows():
    g    = int(row["generator"])
    role = row["role"]
    dev  = row["deviation_MW"]
    diff = row["total_profit_2p"] - row["total_profit_1p"]
    if role == "":
        continue  # neutral, skip
    direction = "▼ causes shortage" if dev < 0 else ("▲ helps system" if dev > 0 else "")
    print(f"  G{g} [{role}]: Δ={dev:+.2f} MW  {direction}")
    print(f"       1-price total: {row['total_profit_1p']:.2f} $  |  "
          f"2-price total: {row['total_profit_2p']:.2f} $  |  diff: {diff:+.2f} $")
 
print("\n--- Wind Farms (BRPs) ---")
for _, row in df_wind5.iterrows():
    w    = int(row["wind"])
    dev  = row["deviation_MW"]
    diff = row["total_profit_2price"] - row["total_profit_1price"]
    direction = "▼ causes shortage" if dev < 0 else "▲ helps system"
    print(f"  W{w}: Δ={dev:+.2f} MW  [{direction}]")
    print(f"       1-price total: {row['total_profit_1price']:.2f} $  |  "
          f"2-price total: {row['total_profit_2price']:.2f} $  |  diff: {diff:+.2f} $")
 

from src.postprocess import step5_build_comparison_table
 
df_comparison5 = step5_build_comparison_table(df_gen5, df_wind5)
 
print("\n=== GROUPED SUMMARY: ONE-PRICE vs TWO-PRICE ===")
print(df_comparison5[["group","members","da_profit",
                       "bal_settlement_1p","total_profit_1p",
                       "bal_settlement_2p","total_profit_2p"]].round(2).to_string(index=False))
 
print("\n--- Implications ---")
for _, row in df_comparison5.iterrows():
    print(f"  {row['group']} ({row['members']}): {row['note']}")
    diff = row["total_profit_2p"] - row["total_profit_1p"]
    if abs(diff) > 1e-6:
        print(f"    → 2-price vs 1-price: {diff:+.2f} $")git 

SyntaxError: invalid syntax (145612892.py, line 64)

# STEP 6

In [ ]:
from src.models import Step6aInputData, Step6aReserveMarket, Step6bInputData, Step6bDAMarket
from src.postprocess import step6_build_summary_tables

PARAMETERS AND DEMAND

In [ ]:
UP_RES_PRICE_PCT  = 0.25   # upward reserve offer = 25% of c_g  $/MW
DN_RES_PRICE_PCT  = 0.40   # downward reserve offer = 40% of c_g $/MW
UP_RES_CAP_PCT    = 0.80   # max upward reserve = 80% of headroom
DN_RES_CAP_PCT    = 0.20   # max downward reserve = 20% of DA dispatch
UP_RES_REQ_PCT    = 0.15   # system upward requirement = 15% of demand
DN_RES_REQ_PCT    = 0.10   # system downward requirement = 10% of demand

# Total demand from Step 1
total_demand = sum(model.results.d[n] for n in inp.LOAD_BUSES)
R_up_req = UP_RES_REQ_PCT * total_demand
R_dn_req = DN_RES_REQ_PCT * total_demand

print(f"Total demand (H=19):       {total_demand:.2f} MW")
print(f"Required upward reserve:   {R_up_req:.2f} MW  (15% of demand)")
print(f"Required downward reserve: {R_dn_req:.2f} MW  (10% of demand)")

Total demand (H=19):       2650.50 MW
Required upward reserve:   397.57 MW  (15% of demand)
Required downward reserve: 265.05 MW  (10% of demand)


RESERVE OFFERS

In [ ]:
GENERATORS_UP_RES = []
GENERATORS_DN_RES = []
up_res_price = {}
dn_res_price = {}
R_up_max = {}
R_dn_max = {}

for g in inp.GENERATORS:
    pG_DA  = model.results.pG[g]
    Pmax   = inp.generator_pmax[g]
    c_g    = inp.generator_cost[g]

    headroom = Pmax - pG_DA     # available upward capacity in Step 1
    footroom = pG_DA            # available downward capacity in Step 1

    up_cap = UP_RES_CAP_PCT * headroom
    dn_cap = DN_RES_CAP_PCT * footroom

    if up_cap > 1e-6:
        GENERATORS_UP_RES.append(g)
        up_res_price[g] = UP_RES_PRICE_PCT * c_g
        R_up_max[g]     = up_cap

    if dn_cap > 1e-6:
        GENERATORS_DN_RES.append(g)
        dn_res_price[g] = DN_RES_PRICE_PCT * c_g
        R_dn_max[g]     = dn_cap

print("\nUpward reserve offer stack:")
for g in sorted(GENERATORS_UP_RES, key=lambda g: up_res_price[g]):
    print(f"  G{g}: {up_res_price[g]:.4f} $/MW,  max={R_up_max[g]:.2f} MW")

print("\nDownward reserve offer stack:")
for g in sorted(GENERATORS_DN_RES, key=lambda g: dn_res_price[g], reverse=True):
    print(f"  G{g}: {dn_res_price[g]:.4f} $/MW,  max={R_dn_max[g]:.2f} MW")


Upward reserve offer stack:
  G1: 3.3300 $/MW,  max=42.25 MW
  G2: 3.3300 $/MW,  max=121.60 MW
  G3: 5.1750 $/MW,  max=280.00 MW
  G4: 5.2325 $/MW,  max=472.80 MW
  G5: 6.5275 $/MW,  max=48.00 MW

Downward reserve offer stack:
  G1: 5.3280 $/MW,  max=19.84 MW
  G12: 4.3560 $/MW,  max=70.00 MW
  G6: 4.2080 $/MW,  max=31.00 MW
  G7: 4.2080 $/MW,  max=31.00 MW
  G11: 4.2080 $/MW,  max=62.00 MW
  G8: 2.4080 $/MW,  max=80.00 MW
  G9: 2.1880 $/MW,  max=80.00 MW
  G10: 0.0000 $/MW,  max=60.00 MW


STAGE 1: CLEAR RESERVE MARKET

In [ ]:
inp6a = Step6aInputData(
    GENERATORS         = inp.GENERATORS,
    GENERATORS_UP_RES  = GENERATORS_UP_RES,
    GENERATORS_DN_RES  = GENERATORS_DN_RES,
    up_res_price       = up_res_price,
    dn_res_price       = dn_res_price,
    R_up_max           = R_up_max,
    R_dn_max           = R_dn_max,
    R_up_req           = R_up_req,
    R_dn_req           = R_dn_req,
)

model6a = Step6aReserveMarket(inp6a, output_flag=0)
model6a.run()

print(f"\n=== RESERVE MARKET RESULTS ===")
print(f"Upward reserve price  (λ_up_res): {model6a.results.lambda_up_res:.4f} $/MW")
print(f"Downward reserve price (λ_dn_res): {model6a.results.lambda_dn_res:.4f} $/MW")
print(f"Total reserve procurement cost:   {model6a.results.objective_value:.2f} $")

print("\nUpward reserve commitments:")
for g in sorted(GENERATORS_UP_RES):
    r = model6a.results.r_up[g]
    if r > 1e-6:
        print(f"  G{g}: {r:.2f} MW")

print("\nDownward reserve commitments:")
for g in sorted(GENERATORS_DN_RES):
    r = model6a.results.r_dn[g]
    if r > 1e-6:
        print(f"  G{g}: {r:.2f} MW")


=== RESERVE MARKET RESULTS ===
Upward reserve price  (λ_up_res): 5.1750 $/MW
Downward reserve price (λ_dn_res): 4.2080 $/MW
Total reserve procurement cost:   2312.39 $

Upward reserve commitments:
  G1: 42.25 MW
  G2: 121.60 MW
  G3: 233.72 MW

Downward reserve commitments:
  G6: 31.00 MW
  G7: 14.05 MW
  G8: 80.00 MW
  G9: 80.00 MW
  G10: 60.00 MW


STAGE 2: BUILD DA INPUT WITH RESERVE CONSTRAINTS

In [ ]:
# Full r_up and r_dn dicts for all generators (0 if not a provider)
r_up_all = {g: model6a.results.r_up.get(g, 0.0) for g in inp.GENERATORS}
r_dn_all = {g: model6a.results.r_dn.get(g, 0.0) for g in inp.GENERATORS}

inp6b = Step6bInputData(
    GENERATORS      = inp.GENERATORS,
    WINDS           = inp.WINDS,
    LOAD_BUSES      = inp.LOAD_BUSES,
    generator_cost  = inp.generator_cost,
    generator_pmax  = inp.generator_pmax,
    wind_avail      = inp.wind_avail,
    demand_pmax     = inp.demand_pmax,
    demand_bid      = inp.demand_bid,
    r_up            = r_up_all,
    r_dn            = r_dn_all,
)


model6b = Step6bDAMarket(inp6b, output_flag=0)
model6b.run()

print(f"\n=== DAY-AHEAD MARKET RESULTS (after reserve) ===")
print(f"DA price Step 1 (no reserve): {model.results.market_price:.4f} $/MWh")
print(f"DA price Step 6 (with reserve): {model6b.results.market_price:.4f} $/MWh")
print(f"Change: {model6b.results.market_price - model.results.market_price:+.4f} $/MWh")


=== DAY-AHEAD MARKET RESULTS (after reserve) ===
DA price Step 1 (no reserve): 13.3200 $/MWh
DA price Step 6 (with reserve): 13.3200 $/MWh
Change: +0.0000 $/MWh


BUILDING TABLES AND DISPLAYING RESULTS

In [ ]:
df_reserve6, df_gen6, df_wind6, df_dem6, totals6 = step6_build_summary_tables(
    res6a      = model6a.results,
    inp6a      = inp6a,
    res6b      = model6b.results,
    inp6b      = inp6b,
    res_step1  = model.results,
    inp_step1  = inp,
)

print("\n=== RESERVE COMMITMENTS & REVENUES ===")
print(df_reserve6.round(3).to_string(index=False))

print("\n=== GENERATOR DISPATCH & PROFITS: STEP 1 vs STEP 6 ===")
print(df_gen6[["generator","marginal_cost","pG_step1_MW","pG_new_MW",
               "r_up_MW","r_dn_MW",
               "da_profit_step1","da_profit_new",
               "reserve_revenue","total_profit_new"]].round(2).to_string(index=False))

print("\n=== WIND FARM PROFITS: STEP 1 vs STEP 6 ===")
print(df_wind6.round(2).to_string(index=False))

print("\n=== DEMAND UTILITY: STEP 1 vs STEP 6 ===")
print(df_dem6.round(2).to_string(index=False))

print("\n=== KEY TOTALS ===")
print(pd.Series(totals6).round(4).to_string())


=== RESERVE COMMITMENTS & REVENUES ===
 generator  up_offer_price  R_up_max_MW  r_up_committed_MW  up_reserve_revenue  dn_offer_price  R_dn_max_MW  r_dn_committed_MW  dn_reserve_revenue  total_reserve_revenue
         1           3.330       42.252             42.252             218.652           5.328       19.837               0.00               0.000                218.652
         2           3.330      121.600            121.600             629.280             NaN        0.000               0.00               0.000                629.280
         3           5.175      280.000            233.723            1209.519             NaN        0.000               0.00               0.000               1209.519
         4           5.232      472.800              0.000               0.000             NaN        0.000               0.00               0.000                  0.000
         5           6.528       48.000              0.000               0.000             NaN        0.000   

COMPARING RESULTS ON HOW THE RESERVE MARKET AFFECTS THE DAY-AHEAD OUTCOME

In [ ]:
print("\n=== PRICE IMPACT ANALYSIS ===")
λ_old = totals6["da_price_step1"]
λ_new = totals6["da_price_new"]
λ_up  = totals6["lambda_up_res"]
λ_dn  = totals6["lambda_dn_res"]

print(f"Step 1 DA price:     {λ_old:.4f} $/MWh")
print(f"Step 6 DA price:     {λ_new:.4f} $/MWh  (change: {λ_new - λ_old:+.4f})")
print(f"Upward reserve price:  {λ_up:.4f} $/MW")
print(f"Downward reserve price: {λ_dn:.4f} $/MW")
print(f"\nGenerators whose dispatch changed:")
changed = df_gen6[abs(df_gen6["dispatch_change_MW"]) > 1e-3]
for _, row in changed.iterrows():
    print(f"  G{int(row['generator'])}: {row['pG_step1_MW']:.2f} → {row['pG_new_MW']:.2f} MW "
          f"(Δ={row['dispatch_change_MW']:+.2f} MW)  "
          f"r_up={row['r_up_MW']:.2f}, r_dn={row['r_dn_MW']:.2f}")

print(f"\nTotal system cost comparison:")
print(f"  Step 1 (energy only):              {totals6['total_system_cost_step1']:.2f} $")
print(f"  Step 6 (reserve + energy):         {totals6['total_system_cost']:.2f} $")
print(f"  Increase due to reserve:           "
      f"{totals6['total_system_cost'] - totals6['total_system_cost_step1']:+.2f} $")



=== PRICE IMPACT ANALYSIS ===
Step 1 DA price:     13.3200 $/MWh
Step 6 DA price:     13.3200 $/MWh  (change: +0.0000)
Upward reserve price:  5.1750 $/MW
Downward reserve price: 4.2080 $/MW

Generators whose dispatch changed:

Total system cost comparison:
  Step 1 (energy only):              16251.05 $
  Step 6 (reserve + energy):         18563.45 $
  Increase due to reserve:           +2312.39 $
